# Evaluating Multiple LM Outputs (External)

In [1]:
# imports
import json
import pandas as pd
import importlib.util
import sys
from os.path import join
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion

In [2]:
# load files
analysis_subdir_path_1 = "analysis_output_1"
analysis_subdir_path_2 = "analysis_output_2"

multirun_filename_1 = "multirun_analyses.json"
multirun_filename_2 = "multirun_analyses.json"

# use both files to get analysis code paths
multirun_path_1 = join(analysis_subdir_path_1, multirun_filename_1)
multirun_path_2 = join(analysis_subdir_path_2, multirun_filename_2)

with open(multirun_path_1, "r") as file:
    multirun_analyses_1 = json.load(file)

with open(multirun_path_2, "r") as file:
    multirun_analyses_2 = json.load(file)

num_analyses_1 = multirun_analyses_1['n']
num_analyses_2 = multirun_analyses_2['n']

analysis_code_filenames_1 = [f"llm_analysis_{i}.py" for i in range(num_analyses_1)]
analysis_code_filenames_2 = [f"llm_analysis_{i}.py" for i in range(num_analyses_2)]

analysis_code_paths_1 = [join(analysis_subdir_path_1, filename)
                         for filename in analysis_code_filenames_1]

analysis_code_paths_2 = [join(analysis_subdir_path_2, filename)
                         for filename in analysis_code_filenames_2]

In [3]:
llm_provider = "openai"
llm_model = "gpt-5-mini"
llm_assistant = llm(provider=llm_provider, model=llm_model)

[2025-11-24 02:18:57.40][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [4]:
features_1 = format_features(multirun_analyses_1, num_analyses_1, llm_assistant)
features_2 = format_features(multirun_analyses_2, num_analyses_2, llm_assistant)

[2025-11-24 02:18:58.65][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:19:07.27][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  8.62 seconds
[2025-11-24 02:19:07.28][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:19:07.30][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:19:16.48][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  9.18 seconds
[2025-11-24 02:19:16.48][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:19:16.50][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:19:23.28][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.78 

In [5]:
model_info_1 = format_model_info(multirun_analyses_1, num_analyses_1, llm_assistant)
model_info_2 = format_model_info(multirun_analyses_2, num_analyses_2, llm_assistant)

[2025-11-24 02:26:03.74][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:26:20.36][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  16.62 seconds
[2025-11-24 02:26:20.36][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:26:20.39][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:26:44.17][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  23.78 seconds
[2025-11-24 02:26:44.17][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:26:44.18][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:26:57.96][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  13.

In [6]:
# load dataset, need more user-friendly input method later
dataset_name = multirun_analyses_1['dataset_name']
dataset_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                    "datasets", dataset_name, "data.csv")

data = pd.read_csv(dataset_path)

In [7]:
transform_functions_1 = {}
transform_functions_2 = {}
model_functions_1 = {}
model_functions_2 = {}

# ----- Loop for first set -----
for i, analysis_code_path in enumerate(analysis_code_paths_1):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_{i}"] = module
    spec.loader.exec_module(module)
    
    transform_functions_1[i] = module.transform
    model_functions_1[i] = module.model

# ----- Loop for second set -----
for i, analysis_code_path in enumerate(analysis_code_paths_2):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_2_{i}"] = module
    spec.loader.exec_module(module)

    transform_functions_2[i] = module.transform
    model_functions_2[i] = module.model


In [8]:
transformed_datasets_1 = {}
for i, transform_func in transform_functions_1.items():
    try:
        transformed_datasets_1[i] = transform_func(data.copy())
        print(f"[Transform 1-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform 1-{i}] Failed with error: {e}")
        transformed_datasets_1[i] = None

transformed_datasets_2 = {}
for i, transform_func in transform_functions_2.items():
    try:
        transformed_datasets_2[i] = transform_func(data.copy())
        print(f"[Transform 2-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform 2-{i}] Failed with error: {e}")
        transformed_datasets_2[i] = None


model_results_1 = {}
for i, model_func in model_functions_1.items():
    try:
        if transformed_datasets_1[i] is None:
            print(f"[Model 1-{i}] Skipping — transform failed.")
            continue

        model_results_1[i] = model_func(transformed_datasets_1[i].copy())
        print(f"[Model 1-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Model 1-{i}] Failed with error: {e}")
        model_results_1[i] = None

model_results_2 = {}
for i, model_func in model_functions_2.items():
    try:
        if transformed_datasets_2[i] is None:
            print(f"[Model 2-{i}] Skipping — transform failed.")
            continue

        model_results_2[i] = model_func(transformed_datasets_2[i].copy())
        print(f"[Model 2-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Model 2-{i}] Failed with error: {e}")
        model_results_2[i] = None


[Transform 1-0] Completed successfully.
[Transform 1-1] Completed successfully.
[Transform 1-2] Completed successfully.
[Transform 2-0] Completed successfully.
[Transform 2-1] Completed successfully.
[Transform 2-2] Completed successfully.


/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 1-0] Completed successfully.


/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 1-1] Completed successfully.


/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:2014: RuntimeWarning: divide by zero encountered in divide
  self.het_scale = (self.wresid / (1 - h))**2
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 1-2] Completed successfully.
[Model 2-0] Completed successfully.
[Model 2-1] Completed successfully.
[Model 2-2] Completed successfully.


In [9]:
final_answer_code_1 = {}
final_answer_code_2 = {}

info_json_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                      "datasets", dataset_name, "info.json")
with open(info_json_path, "r") as file:
    info_json = json.load(file)

task = info_json['research_questions']

for i in range(num_analyses_1):

    independent_variable = features_1[i]['independent_variables']
    dependent_variable = features_1[i]['response_variables']

    model_code = multirun_analyses_1['analyses'][str(i)]['m_code']
    model_output = model_results_1[i]

    final_answer_code_1[i] = write_final_answer_code(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        model_output
    )

for i in range(num_analyses_2):

    independent_variable = features_2[i]['independent_variables']
    dependent_variable = features_2[i]['response_variables']

    model_code = multirun_analyses_2['analyses'][str(i)]['m_code']
    model_output = model_results_2[i]

    final_answer_code_2[i] = write_final_answer_code(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        model_output
    )


[2025-11-24 02:27:49.06][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:28:33.15][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  44.09 seconds
[2025-11-24 02:28:33.16][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:28:33.20][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:28:56.67][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  23.46 seconds
[2025-11-24 02:28:56.68][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:28:56.72][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:29:26.69][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  29.

In [10]:
print(final_answer_code_1)
print(final_answer_code_2)

{0: 'def extract_final_answer(model_output):\n    """\n    Extracts coefficients, standard errors, p-values, 95% CIs, and interpretable effect sizes\n    for the main independent variables in the supplied model_output dict.\n\n    Returns a dict with keys:\n      - "object": dict of extracted numeric results (per model)\n      - "description": human-readable summary interpreting whether the results support\n                       the hypothesis that more feminine names are associated with fewer fatalities.\n    """\n    import numpy as np\n\n    results = {}\n    summary_lines = []\n\n    # Helper to safely extract info for a variable from a results object\n    def extract_from_result(res, var_name, model_type):\n        out = {"variable": var_name, "model_type": model_type}\n        try:\n            params = getattr(res, "params")\n            if var_name not in params.index:\n                out["error"] = f"Variable \'{var_name}\' not present in model parameters."\n                

In [11]:
final_answer_functions_1 = {}

for i in range(num_analyses_1):
    namespace = {}

    compiled_code = compile(
        final_answer_code_1[i],
        f"<final_answer_code_1_{i}>",
        "exec"
    )
    exec(compiled_code, namespace)

    final_answer_functions_1[i] = namespace['extract_final_answer']

final_answers_1 = [
    final_answer_functions_1[i](model_results_1[i])
    for i in range(num_analyses_1)
]


final_answer_functions_2 = {}

for i in range(num_analyses_2):
    namespace = {}

    compiled_code = compile(
        final_answer_code_2[i],
        f"<final_answer_code_2_{i}>",
        "exec"
    )
    exec(compiled_code, namespace)

    final_answer_functions_2[i] = namespace['extract_final_answer']

final_answers_2 = [
    final_answer_functions_2[i](model_results_2[i])
    for i in range(num_analyses_2)
]


In [12]:
conclusions_1 = {}

for i in range(num_analyses_1):
    independent_variable = features_1[i]['independent_variables']
    dependent_variable = features_1[i]['response_variables']

    model_code = multirun_analyses_1['analyses'][str(i)]['m_code']
    interpretation_code = final_answer_code_1.get(i, None)
    interpretation_output = final_answers_1[i] if i < len(final_answers_1) else None

    conclusions_1[i] = make_conclusion(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        interpretation_code,
        interpretation_output
    )


conclusions_2 = {}

for i in range(num_analyses_2):
    independent_variable = features_2[i]['independent_variables']
    dependent_variable = features_2[i]['response_variables']

    model_code = multirun_analyses_2['analyses'][str(i)]['m_code']

    interpretation_code = final_answer_code_2.get(i, None)
    interpretation_output = final_answers_2[i] if i < len(final_answers_2) else None

    conclusions_2[i] = make_conclusion(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        interpretation_code,
        interpretation_output
    )


[2025-11-24 02:30:51.24][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)


[2025-11-24 02:30:57.57][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.33 seconds
[2025-11-24 02:30:57.57][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:30:57.60][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:31:03.96][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.36 seconds
[2025-11-24 02:31:03.97][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:31:03.99][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:31:09.57][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.58 seconds
[2025-11-24 02:31:09.57][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2

In [13]:
llm_judge = llm(provider=llm_provider, model=llm_model)
data_head = data.head(10)

[2025-11-24 02:31:29.27][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [14]:
print()

In [15]:
judge_system_prompt = (
    "You are a meticulous research design evaluator. "
    "Your role is to compare two experimental trials methodologically **and interpretively**.\n\n"
    "You will go through the following reasoning plan step-by-step (internally):\n"
    "1. Understand the research question and dataset context.\n"
    "2. Examine independent, control, and response variables for both trials.\n"
    "3. Analyze the model specifications for structural or methodological similarity.\n"
    "4. Focus more on the content, less on the format.\n"
    "5. Assess whether the trials' conclusions are logically consistent given their setups.\n"
    "6. Detect whether either input is None, invalid, erroneous, or incomplete.\n"
    "   - If **one trial** shows errors or missing components but the other is valid, "
    "     impose a **strong penalty** (reduce all category scores by at least 1 point, "
    "     and cap overall similarity at 2).\n"
    "7. Synthesize your evaluation across all components.\n"
    "8. Output a numerical rating for each category.\n\n"
    "DO NOT include your reasoning — only the final JSON object.\n\n"
    "Scoring scale:\n"
    "1 = completely different\n"
    "2 = somewhat different\n"
    "3 = moderately similar\n"
    "4 = very similar\n"
    "5 = almost identical\n\n"
    "Return output **strictly in JSON format**:\n"
    "{\n"
    "  \"independent_variables\": <number>,\n"
    "  \"control_variables\": <number>,\n"
    "  \"response_variables\": <number>,\n"
    "  \"model_specification\": <number>,\n"
    "  \"conclusions\": <number>,\n"
    "  \"overall_similarity\": <number>\n"
    "}"
)


def make_judge_prompt(task, data_head, featA, featB, modelA, modelB, conclA, conclB):
    return (
        f"Research Question / Context:\n{task}\n\n"
        "Here is a sample of the dataset to understand the structure and variables:\n"
        f"{data_head}\n\n"
        "Compare the two trials methodologically and interpretively based on the provided variables, model specifications, and conclusions.\n\n"
        "==================== TRIAL A ====================\n\n"
        "Independent Variables:\n"
        f"{featA['independent_variables']}\n\n"
        "Control Variables:\n"
        f"{featA.get('control_variables')}\n\n"
        "Response Variables:\n"
        f"{featA['response_variables']}\n\n"
        "Model Specification:\n"
        f"{modelA}\n\n"
        "Conclusion:\n"
        f"{conclA}\n\n"
        "==================== TRIAL B ====================\n\n"
        "Independent Variables:\n"
        f"{featB['independent_variables']}\n\n"
        "Control Variables:\n"
        f"{featB.get('control_variables')}\n\n"
        "Response Variables:\n"
        f"{featB['response_variables']}\n\n"
        "Model Specification:\n"
        f"{modelB}\n\n"
        "Conclusion:\n"
        f"{conclB}\n\n"
        "Now, following your reasoning plan, provide similarity ratings as JSON only."
    )


In [16]:
judge_results = {}

num_comparisons = min(num_analyses_1, num_analyses_2)

for i in range(num_comparisons):

    user_prompt = make_judge_prompt(
        task, 
        data_head,
        features_1[i], features_2[i],
        model_info_1[i], model_info_2[i],
        conclusions_1[i], conclusions_2[i]
    )

    # call LLM judge
    result = llm_judge.generate([
        {"role": "system", "content": judge_system_prompt},
        {"role": "user", "content": user_prompt}
    ])

    judge_results[i] = result


[2025-11-24 02:31:29.97][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)


[2025-11-24 02:31:36.80][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.84 seconds
[2025-11-24 02:31:36.80][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:31:36.86][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:31:48.48][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  11.62 seconds
[2025-11-24 02:31:48.48][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:31:48.53][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:32:00.70][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  12.17 seconds
[2025-11-24 02:32:00.71][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [17]:
judge_results

{0: TextGenResponse(text=[Message(role='assistant', content='{\n  "independent_variables": 5,\n  "control_variables": 4,\n  "response_variables": 4,\n  "model_specification": 3,\n  "conclusions": 5,\n  "overall_similarity": 4\n}')], config=TextGenConfig(model='gpt-5-mini', n=1, temperature=0.8, max_tokens=None, top_p=None, top_k=None, run_config=None, stop_sequences=None, frequency_penalty=0.0, presence_penalty=0.0), api_elapsed_time=6.83745265007019, cache_elapsed_time=None, from_cache=False, response=ChatCompletion(id='chatcmpl-CfZNicghA3QZimqz8vWhB6HMN7jsT', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "independent_variables": 5,\n  "control_variables": 4,\n  "response_variables": 4,\n  "model_specification": 3,\n  "conclusions": 5,\n  "overall_similarity": 4\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1764023490, model='gpt-5-mini-2025-08-07', object='ch